# Forest Plots for an Environmental Meta-Analysis: N2O Emissions and Crop Yield

**Author:** Mohammad Rabbani · **Source scripts:** [Agricultural_meta-analysis_viz](https://github.com/Rabbani-bot/Agricultural_meta-analysis_viz)

## Abstract
This notebook synthesises published meta-analyses on how nitrogen-management practices affect **soil N2O emissions** (10 meta-analyses, 1,899 observations) and **crop yield** (9 meta-analyses, 1,340 observations). Effects are on the log response-ratio scale (ln RR; negative = reduction), back-transformed to percentage change for interpretation.

**Quality control.** Every confidence interval is checked against its mean and standard error before plotting. Six N2O lower limits did not match their own mean and SE (they duplicated the yield panel's lower limits, a transcription error), so all intervals are recomputed as mean ± 1.96 × SE. An independent random-effects re-pooling then verifies the reported grand means.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "font.size": 9})

n2o = pd.DataFrame({
    'Study': ['Cai et al (2017) ', 'Ma et al (2023)', 'Fan et al (2022)', 'Liao et al (2021)', 'Tufail et al (2023)', 'Thapa et al (2016)', 'Di Wu et al (2021)', 'Ribeiro et al (2023)', 'Soares et al (2023)', 'Wang et al (2024)', 'Grand mean'],
    'L1': [-0.002033886, -0.085681899, 0.006026948, 0.022212993, -0.120704656, -0.113818774, -0.6698780909, -0.9395422339, -0.9592365826, -0.6647951034, -0.6802914917],
    'L2': [-0.2075265192, -0.1464455668, -0.4965336415, -0.4008631672, -0.554127865, -0.3431529298, -0.2991557559, -0.4292441277, -0.7082604559, -0.4461406026, -0.4344652722],
    'Mean': [-0.3021983951, -0.3270452054, -0.5352054846, -0.6792955445, -0.6415707825, -0.5643638738, -0.4845169234, -0.6843931808, -0.8337485192, -0.555467853, -0.5573783819],
    'Numbers': [138, 217, 656, 38, 180, 123, 60, 74, 269, 144, 1899],
    'SE': [0.04830197747, 0.0921426777, 0.01973053218, 0.1420573353, 0.04461373342, 0.1128627266, 0.09457202422, 0.1301780883, 0.06402452211, 0.05577920939, 0.0627107703]
})
yld = pd.DataFrame({
    'Study': ['Ma et al (2023)', 'Cai et al (2017)', 'Fan et al (2022)', 'Liao et al (2021)', 'Tufail et al (2023)', 'Thapa et al (2016)', 'Abalos et al (2014)', 'Ribeiro et al (2023)', 'Wang et al (2024)', 'Grand mean'],
    'L1': [-0.085681899, -0.002033886001, 0.006026947998, 0.02221299303, -0.120704656, -0.1138187743, 0.02351739823, -0.03722381183, 0.03039437643, -0.02032692129],
    'L2': [0.1889940519, 0.1541452923, 0.0566177146, 0.1560828779, 0.01442945, 0.1629045606, 0.1158370324, 0.101194414, 0.09394816161, 0.104084878],
    'Mean': [0.05165607648, 0.07605570316, 0.0313223313, 0.08914793545, -0.053137603, 0.02454289314, 0.06967721532, 0.03198530107, 0.06217126902, 0.04187897838],
    'Numbers': [68, 92, 328, 39, 125, 81, 275, 109, 223, 1340],
    'SE': [0.07007039563, 0.03984162712, 0.01290580781, 0.03415048082, 0.03447298622, 0.07059268746, 0.02355092708, 0.03531077189, 0.0162127003, 0.03173770391]
})
for d in (n2o, yld):
    d["Study"] = d["Study"].str.strip()
n2o.tail(3)

## 1. Confidence-interval integrity check

In [ ]:
def check(d, label):
    out = d.assign(lo_calc=d.Mean - 1.96 * d.SE, hi_calc=d.Mean + 1.96 * d.SE)
    out["reported_lo_ok"] = np.isclose(out.L1, out.lo_calc, atol=0.01)
    out["reported_hi_ok"] = np.isclose(out.L2, out.hi_calc, atol=0.01)
    bad = out[~(out.reported_lo_ok & out.reported_hi_ok)]
    print(f"{label}: {len(bad)} of {len(out)} rows inconsistent with mean ± 1.96·SE")
    return out, bad

n2o_chk, n2o_bad = check(n2o, "N2O"); yld_chk, yld_bad = check(yld, "Yield")
dup = n2o_bad.merge(yld[["Study", "L1"]], on="Study", suffixes=("_n2o", "_yield"))
dup["identical_to_yield_L1"] = np.isclose(dup.L1_n2o, dup.L1_yield)
dup[["Study", "Mean", "SE", "L1_n2o", "lo_calc", "L1_yield", "identical_to_yield_L1"]].round(3)

The inconsistent N2O lower limits are identical to the yield panel's lower limits for the same studies, which confirms a transcription error. Means, standard errors and upper limits are internally consistent, so the corrected intervals are fully recoverable: **all intervals below use mean ± 1.96 × SE**.

## 2. Independent re-pooling of the grand means (DerSimonian–Laird random effects)

In [ ]:
def dl_pool(y, se):
    w = 1 / se**2; fe = np.sum(w * y) / w.sum()
    Q = np.sum(w * (y - fe) ** 2); df_ = len(y) - 1
    tau2 = max(0.0, (Q - df_) / (w.sum() - (w**2).sum() / w.sum()))
    wr = 1 / (se**2 + tau2); mu = np.sum(wr * y) / wr.sum(); s = np.sqrt(1 / wr.sum())
    I2 = max(0.0, (Q - df_) / Q) * 100 if Q > 0 else 0.0
    return dict(k=len(y), pooled=mu, ci_low=mu - 1.96 * s, ci_high=mu + 1.96 * s, tau2=tau2, Q=Q,
                p_Q=stats.chi2.sf(Q, df_), I2=I2)

rows = []
for label, d in [("N2O", n2o), ("Yield", yld)]:
    st = d[d.Study != "Grand mean"]; gm = d[d.Study == "Grand mean"].iloc[0]
    r = dl_pool(st.Mean.to_numpy(), st.SE.to_numpy())
    rows.append(dict(outcome=label, reported_grand_mean=gm.Mean, **r))
pool = pd.DataFrame(rows).set_index("outcome")
pool["reported_%"] = 100 * (np.exp(pool.reported_grand_mean) - 1)
pool["repooled_%"] = 100 * (np.exp(pool.pooled) - 1)
pool.round(3)

## 3. Corrected forest plots

In [ ]:
def forest(ax, d, title, color):
    st = d[d.Study != "Grand mean"].copy(); gm = d[d.Study == "Grand mean"].iloc[0]
    st["lo"], st["hi"] = st.Mean - 1.96 * st.SE, st.Mean + 1.96 * st.SE
    st = st.sort_values("Mean").reset_index(drop=True)
    y = np.arange(len(st))
    ax.errorbar(st.Mean, y, xerr=[st.Mean - st.lo, st.hi - st.Mean], fmt="none", ecolor="#8A94A6", capsize=2)
    ax.scatter(st.Mean, y, s=12 + st.Numbers / 4, color="white", edgecolor=color, zorder=3)
    g_lo, g_hi = gm.Mean - 1.96 * gm.SE, gm.Mean + 1.96 * gm.SE
    gy = len(st) + 0.8
    ax.fill([g_lo, gm.Mean, g_hi, gm.Mean], [gy, gy + 0.35, gy, gy - 0.35], color=color)
    ax.axhline(len(st) + 0.2, color="#BBB", lw=0.6)
    ax.axvline(0, color="#C0392B", ls="--", lw=0.8)
    labels = list(st.Study) + [f"Grand mean ({100*(np.exp(gm.Mean)-1):+.1f}%)"]
    ax.set_yticks(list(y) + [gy]); ax.set_yticklabels(labels); ax.invert_yaxis()
    for yi, n in zip(y, st.Numbers):
        ax.annotate(f"n={n}", (1.01, yi), xycoords=("axes fraction", "data"), va="center", fontsize=7, color="#555")
    ax.set_xlabel("Effect size, ln RR (95% CI)"); ax.set_title(title, loc="left")
    sec = ax.secondary_xaxis("top", functions=(lambda x: 100 * (np.exp(x) - 1), lambda p: np.log(1 + p / 100)))
    sec.set_xlabel("% change")

fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
forest(axes[0], n2o, "N2O emissions (k = 10, n = 1,899)", "#1D4E89")
forest(axes[1], yld, "Crop yield (k = 9, n = 1,340)", "#2E8B57")
fig.tight_layout(); fig.savefig("forest_plots_corrected.png", dpi=300, bbox_inches="tight"); plt.show()

## 4. Interpretation
- **N2O.** All ten meta-analyses report reductions; the pooled effect corresponds to roughly a **40% reduction** in N2O emissions, with substantial between-study heterogeneity (see *I*² above).
- **Yield.** The pooled yield effect is small and positive (about **+4 to +5%**). The reported grand-mean interval spans zero, while the inverse-variance re-pooling gives a 95% CI of roughly +2% to +7%: either way, the practices cut emissions **with no yield penalty**, and possibly a modest gain.
- **Verification.** The independent random-effects re-pooling reproduces the reported grand means closely, so the conclusions are robust to the interval correction (which changes error bars, not means).

## 5. Limitations
- The inputs are themselves meta-analyses that may share primary studies, so their estimates are not fully independent; the pooled intervals are therefore optimistic.
- Observation counts (*n*) are reported for context but are not used as weights; weighting uses inverse variance.